## Filtering data

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

import os

from estnltk_core.converters.json_importer import json_to_text
from estnltk_core.converters.json_exporter import text_to_json

In [ ]:
SENTENCES = "../data/koondkorpus_sentences/conx_example_sentences_20042026/"
RESULT = "../data/koondkorpus_sentences/el_conx_sentences_cleaned/"

In [ ]:
filenames = os.listdir(SENTENCES)

#### Filtering data (strict)

Initially, sentences were extracted from Estonian Reference Corpus based on more lenient conditions. Now, stricter conditions are applied in order to get rid of noise (sentences containing elative construction candidates that do not match the updated pattern).

In [ ]:
def filter_sentences(source_file_path: str, target_file_path: str, filenames_list: list[str]) -> int:
    n_cleaned_sentences = 0

    for idx, filename in enumerate(filenames_list):

        sentence_text = json_to_text(file=f"{source_file_path}{filename}")

        # looking for elative construction phrase
        words = sentence_text.v172_stanza_syntax
    
        for stanza_word in words:
            # ignoring words that are not nominal modifiers
            if stanza_word.deprel != "nmod":
                continue
        
            morph = stanza_word.morph_analysis
            pos = morph.partofspeech
            form = morph.form

            # excluding instances where nominal modifier isn't a substantive or in singular elative case
            if "S" not in pos or "sg el" not in form:
                continue
            
            # excluding instances where nominal modifier in elative case doesn't have a syntactic parent
            parent = stanza_word.parent_span
            if parent is None:
                continue
            
            # excluding instances where nominal modifier in elative case succeeds its syntactic parent
            if parent.id < stanza_word.id:
                continue
            
            # excluding instances where parent is not a substantive
            parent_pos = parent.morph_analysis.partofspeech
            if "S" not in parent_pos:
                continue
            
            # excluding instances where parent lemma ends with 'mine'
            parent_lemma = parent.morph_analysis.lemma[0]
            if parent_lemma.endswith("mine"):
                continue
            
            # excluding instances where parent is in terminative case
            parent_form = parent.morph_analysis.form
            if "sg ter" in parent_form or "pl ter" in parent_form:
                continue
            
            # saving
            text_to_json(sentence_text, file=f"{target_file_path}{filename}")
            n_cleaned_sentences += 1
            break

    return n_cleaned_sentences

In [ ]:
n_cleaned_sentences = filter_sentences(SENTENCES, RESULT, filenames)

In [10]:
n_cleaned_sentences

89719